In [ ]:
# Required Libraries
import pandas as pd
import requests
from pathlib import Path
from io import BytesIO
from typing import Optional, Union

## 1. Define the Storage Client Class

In [ ]:
class StorageClient:
    """
    A simple storage client for loading data from various sources.
    Supports: Local files, URLs, CSV, Parquet, JSON, Excel
    """
    
    def load_from_local(self, path: str, file_format: Optional[str] = None) -> pd.DataFrame:
        """Load DataFrame from a local file."""
        path = Path(path)
        
        if not path.exists():
            raise FileNotFoundError(f"File not found: {path}")
        
        # Auto-detect format from extension
        if file_format is None:
            file_format = path.suffix.lower().lstrip(".")
        
        # Load based on format
        loaders = {
            "csv": pd.read_csv,
            "parquet": pd.read_parquet,
            "json": pd.read_json,
            "xlsx": pd.read_excel,
            "xls": pd.read_excel,
        }
        
        loader = loaders.get(file_format, pd.read_csv)
        return loader(path)
    
    def load_from_url(self, url: str, file_format: Optional[str] = None) -> pd.DataFrame:
        """Load DataFrame from a URL."""
        # Fetch content
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Auto-detect format from URL
        if file_format is None:
            url_path = url.split("?")[0]
            file_format = Path(url_path).suffix.lower().lstrip(".")
        
        content = BytesIO(response.content)
        
        # Load based on format
        if file_format == "csv":
            return pd.read_csv(content)
        elif file_format == "parquet":
            return pd.read_parquet(content)
        elif file_format == "json":
            return pd.read_json(content)
        else:
            return pd.read_csv(content)  # Default to CSV
    
    def load(self, source: str) -> pd.DataFrame:
        """Universal loader - auto-detects if source is URL or local file."""
        if source.startswith("http://") or source.startswith("https://"):
            print(f"📡 Loading from URL: {source[:50]}...")
            return self.load_from_url(source)
        else:
            print(f"📁 Loading from local: {source}")
            return self.load_from_local(source)

## 2. Test with Local CSV File

In [ ]:
# Initialize the storage client
client = StorageClient()

# Load local CSV
df_local = client.load("../data/demo_sales.csv")

print(f"✅ Loaded {len(df_local)} rows, {len(df_local.columns)} columns")
print(f"\nColumns: {list(df_local.columns)}")
df_local.head()

## 3. Test with URL (Titanic Dataset)

In [ ]:
# Load from URL
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_url = client.load(titanic_url)

print(f"✅ Loaded {len(df_url)} rows, {len(df_url.columns)} columns")
print(f"\nColumns: {list(df_url.columns)}")
df_url.head()

## 4. Quick Data Summary Function

In [ ]:
def quick_summary(df: pd.DataFrame, name: str = "Dataset") -> None:
    """Print a quick summary of the DataFrame."""
    print(f"\n{'='*50}")
    print(f"📊 {name} Summary")
    print(f"{'='*50}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"\nData Types:")
    print(df.dtypes.value_counts().to_string())
    print(f"\nMissing Values:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0].to_string())
    else:
        print("  No missing values!")

# Test it
quick_summary(df_local, "Demo Sales")
quick_summary(df_url, "Titanic")

## ✅ Summary

This module provides:
- `StorageClient.load_from_local()` - Load from local files
- `StorageClient.load_from_url()` - Load from URLs
- `StorageClient.load()` - Universal loader (auto-detects source type)
- Supports: CSV, Parquet, JSON, Excel formats